# 장애인콜택시 대기시간 예측 모델링 - RandomForest Time-based Supply Proxy

이 노트북은 기존 RandomForest 모델링 흐름을 기반으로, 시계열 관점의 공급 proxy를 추가해 대기시간 예측 성능을 확인하는 실험 노트북이다.

## 목적

- 프로젝트 폴더의 `data/processed`에서 전처리 CSV 불러오기
- 임차택시 바로콜 / 특장차 바로콜 승차완료 데이터 생성
- 130분 이후 장시간 대기와 02~06시 새벽 호출을 별도관리 구간으로 분리
- 기존 random split이 아니라 `접수일시` 기준 time-based split 적용
- 차량 위치 데이터가 없는 한계를 보완하기 위해 승차/하차 이력 기반 공급 흐름 proxy 생성
- RandomForest로 다음 3개 피처셋 비교
  1. 기존 Feature Set v2
  2. 기존 Feature Set v2 + 전일 차량운행
  3. 기존 Feature Set v2 + 전일 차량운행 + model_group별 출발구 공급 흐름 proxy

## 핵심 아이디어

현재 호출의 출발구에 대해, 접수시각 이전 30분/60분 동안 같은 차량유형이 해당 구에 하차한 건수를 공급 유입으로 보고, 해당 구에서 승차한 건수를 공급 유출로 본다.

```text
순공급 proxy = 최근 하차 건수 - 최근 승차 건수
```

주의: 이 실험은 time-based split 기준이므로 기존 random split 결과와 절대 수치를 직접 비교하지 않고, 같은 time-based split 안에서 피처 추가 전후 성능 변화를 비교한다.


## 1. 라이브러리 및 로컬 프로젝트 경로 설정

VSCode에서 이 노트북을 실행할 때는 Google Drive 경로를 사용하지 않는다.

현재 프로젝트 구조를 기준으로 아래 CSV를 불러온다.

```text
data/processed/임차택시_대기시간_전처리.csv
data/processed/특장차_대기시간_전처리_접수유형분류.csv
```

노트북 실행 위치가 프로젝트 루트이든 `notebooks_waiting_time` 폴더이든 자동으로 프로젝트 루트를 찾도록 설정한다.

In [1]:
from pathlib import Path
import unicodedata
import platform

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)


# 한글 폰트 설정
if platform.system() == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
elif platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
else:
    plt.rcParams["font.family"] = "NanumGothic"

plt.rcParams["axes.unicode_minus"] = False


In [2]:
# =========================
# 로컬 프로젝트 경로 설정
# =========================

RENTAL_FILENAME = "임차택시_대기시간_전처리.csv"
SPECIAL_FILENAME = "특장차_대기시간_전처리_접수유형분류.csv"


def normalize_text(text):
    return unicodedata.normalize("NFC", str(text))


def find_project_root(start_path=None):
    """
    현재 실행 위치에서 위로 올라가며 data/processed 폴더가 있는 프로젝트 루트를 찾는다.
    """
    start = Path.cwd() if start_path is None else Path(start_path)
    candidates = [start] + list(start.parents)

    for candidate in candidates:
        if (candidate / "data" / "processed").exists():
            return candidate

    raise FileNotFoundError(
        "data/processed 폴더를 찾을 수 없습니다. "
        "VSCode의 현재 작업 폴더가 calltaxi-DA 프로젝트 안인지 확인하세요."
    )


def find_csv_in_processed(processed_dir, filename):
    """
    1순위: data/processed/{filename}
    2순위: 한글 자모 정규화 차이를 고려해 processed 폴더의 CSV 전체에서 파일명 비교
    """
    direct_path = processed_dir / filename

    if direct_path.exists():
        return direct_path

    target_name = normalize_text(filename)
    matches = [
        path
        for path in processed_dir.rglob("*.csv")
        if normalize_text(path.name) == target_name
    ]

    if len(matches) == 0:
        print("processed 폴더에서 찾은 CSV 파일:")
        for path in sorted(processed_dir.rglob("*.csv")):
            print(" -", path)
        raise FileNotFoundError(
            f"{filename} 파일을 찾을 수 없습니다. "
            f"{processed_dir} 안의 파일명을 확인하세요."
        )

    if len(matches) > 1:
        print(f"{filename} 후보가 여러 개입니다. 첫 번째 파일을 사용합니다.")
        for path in matches:
            print(" -", path)

    return matches[0]


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "processed"

RENTAL_PATH = find_csv_in_processed(DATA_DIR, RENTAL_FILENAME)
SPECIAL_PATH = find_csv_in_processed(DATA_DIR, SPECIAL_FILENAME)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("RENTAL_PATH:", RENTAL_PATH)
print("SPECIAL_PATH:", SPECIAL_PATH)
print("RENTAL exists:", RENTAL_PATH.exists())
print("SPECIAL exists:", SPECIAL_PATH.exists())

PROJECT_ROOT: /Users/blaumonde/calltaxi-DA
DATA_DIR: /Users/blaumonde/calltaxi-DA/data/processed
RENTAL_PATH: /Users/blaumonde/calltaxi-DA/data/processed/임차택시_대기시간_전처리.csv
SPECIAL_PATH: /Users/blaumonde/calltaxi-DA/data/processed/특장차_대기시간_전처리_접수유형분류.csv
RENTAL exists: True
SPECIAL exists: True


## 2. 데이터 로드

모델링 대상은 기존과 동일하게 바로콜 승차완료 데이터만 사용한다.

- 임차택시: `임차택시_바로콜여부 == True`, `대기시간분석_포함여부 == True`
- 특장차: `특장차_바로콜_후보여부 == True` 또는 `특장차_접수유형_후보_최종 == "바로콜 후보"`

In [3]:
SEOUL_GU = {
    "강남구", "강동구", "강북구", "강서구", "관악구",
    "광진구", "구로구", "금천구", "노원구", "도봉구",
    "동대문구", "동작구", "마포구", "서대문구", "서초구",
    "성동구", "성북구", "송파구", "양천구", "영등포구",
    "용산구", "은평구", "종로구", "중구", "중랑구",
}


def existing_cols(path, wanted_cols):
    header = pd.read_csv(path, nrows=0).columns.tolist()
    return [col for col in wanted_cols if col in header]


def classify_move_type(row):
    origin = row.get("출발구")
    dest = row.get("목적구")

    origin_in_seoul = origin in SEOUL_GU
    dest_in_seoul = dest in SEOUL_GU

    if origin_in_seoul and dest_in_seoul:
        if origin == dest:
            return "구 내 이동"
        return "구 간 이동"

    if origin_in_seoul and not dest_in_seoul:
        return "서울→서울 외"

    if not origin_in_seoul and dest_in_seoul:
        return "서울 외→서울"

    return "서울 외↔서울 외"


def load_modeling_dataset():
    common_cols = [
        "접수일시", "예정일시", "배차일시", "승차일시", "하차일시", "취소일시",
        "출발구", "출발동", "목적구", "목적동",
        "이용목적", "요금", "승차거리", "차량구분", "장애유형",
        "접수_배차_분", "배차_승차_분", "접수_승차_분",
        "접수_취소_분", "배차_취소_분", "접수승차_날짜차이",
    ]

    rental_cols = common_cols + [
        "예약목적여부",
        "임차택시_바로콜여부",
        "임차택시_장시간예외여부",
        "임차택시_예약성예외여부",
        "임차택시_취소분석유형",
        "대기시간분석_포함여부",
        "대기시간분석_제외사유",
    ]

    special_cols = common_cols + [
        "접수시간대", "접수시간대_HH", "접수요일", "평일주말",
        "세부이동유형", "승차거리_km", "승차거리구간",
        "특장차_접수유형", "특장차_접수유형_분류상태", "특장차_접수유형_메모",
        "접수일자", "예정일자", "취소일자", "접수시", "예정시", "예정시간",
        "취소_접수유형_후보", "_원자료_index",
        "특장차_탑승완료_필수일시존재여부",
        "특장차_탑승완료_시간논리정상여부",
        "심야시간사전예약_후보여부",
        "전일접수_후보여부",
        "특장차_바로콜_후보여부",
        "특장차_접수유형_후보_보완",
        "정기접수_목적후보여부",
        "정기접수_가능패턴여부",
        "동일패턴건수_보완",
        "예정_배차_분", "예정_승차_분",
        "특장차_접수유형_후보_최종",
    ]

    rental = pd.read_csv(
        RENTAL_PATH,
        usecols=existing_cols(RENTAL_PATH, rental_cols),
        low_memory=False,
    )

    special = pd.read_csv(
        SPECIAL_PATH,
        usecols=existing_cols(SPECIAL_PATH, special_cols),
        low_memory=False,
    )

    print("임차택시 원본 shape:", rental.shape)
    print("특장차 원본 shape:", special.shape)

    # datetime 변환
    for frame in [rental, special]:
        for col in [
            "접수일시", "예정일시", "배차일시", "승차일시", "하차일시", "취소일시",
            "접수일자", "예정일자", "취소일자",
        ]:
            if col in frame.columns:
                frame[col] = pd.to_datetime(frame[col], errors="coerce")

    # 임차택시 바로콜 승차완료
    rental_model = rental[
        rental["접수일시"].notna()
        & rental["승차일시"].notna()
        & rental["임차택시_바로콜여부"].fillna(False).astype(bool)
        & rental["대기시간분석_포함여부"].fillna(False).astype(bool)
    ].copy()

    rental_model["model_group"] = "임차택시_바로콜"

    # 특장차 바로콜 승차완료
    special_model = special[
        special["접수일시"].notna()
        & special["승차일시"].notna()
        & (
            special["특장차_바로콜_후보여부"].fillna(False).astype(bool)
            | special["특장차_접수유형_후보_최종"].eq("바로콜 후보")
        )
    ].copy()

    special_model["model_group"] = "특장차_바로콜"

    print("임차택시 바로콜 승차완료:", rental_model.shape)
    print("특장차 바로콜 승차완료:", special_model.shape)

    # 컬럼 맞춰서 결합
    all_cols = sorted(set(rental_model.columns) | set(special_model.columns))

    data = pd.concat(
        [
            rental_model.reindex(columns=all_cols),
            special_model.reindex(columns=all_cols),
        ],
        ignore_index=True,
    )

    # numeric 변환
    numeric_cols = [
        "요금", "승차거리", "승차거리_km",
        "접수_배차_분", "배차_승차_분", "접수_승차_분",
        "접수_취소_분", "배차_취소_분", "접수승차_날짜차이",
        "접수시간대_HH", "접수시", "예정시",
        "예정_배차_분", "예정_승차_분", "동일패턴건수_보완",
    ]

    for col in numeric_cols:
        if col in data.columns:
            data[col] = pd.to_numeric(data[col], errors="coerce")

    # target 생성
    data = data[
        data["접수일시"].notna()
        & data["승차일시"].notna()
        & data["접수_승차_분"].notna()
        & data["접수_승차_분"].ge(0)
    ].copy()

    data["target_min"] = data["접수_승차_분"]

    # 시간 변수
    data["hour"] = data["접수일시"].dt.hour.astype("int16")
    data["dayofweek"] = data["접수일시"].dt.dayofweek.astype("int16")
    data["month"] = data["접수일시"].dt.month.astype("int16")

    data["is_weekend"] = data["dayofweek"].isin([5, 6]).astype("int8")
    data["is_night"] = data["hour"].between(0, 6, inclusive="both").astype("int8")
    data["is_dawn_02_06"] = data["hour"].between(2, 6, inclusive="both").astype("int8")
    data["is_commute"] = (
        data["hour"].between(7, 9, inclusive="both")
        | data["hour"].between(17, 19, inclusive="both")
    ).astype("int8")

    # 승차거리_km 통일
    if "승차거리_km" not in data.columns:
        data["승차거리_km"] = np.nan

    if "승차거리" in data.columns:
        data["승차거리_km"] = data["승차거리_km"].fillna(
            pd.to_numeric(data["승차거리"], errors="coerce")
        )

    # 세부이동유형 보완
    computed_move_type = data.apply(classify_move_type, axis=1)

    if "세부이동유형" in data.columns:
        data["세부이동유형"] = data["세부이동유형"].fillna(computed_move_type)
        data.loc[data["세부이동유형"].astype(str).eq(""), "세부이동유형"] = computed_move_type
    else:
        data["세부이동유형"] = computed_move_type

    # 문자열 결측 처리
    object_cols = data.select_dtypes(include="object").columns

    for col in object_cols:
        data[col] = data[col].fillna("미상").astype(str)

    data = data.sort_values("접수일시").reset_index(drop=True)

    return data

## 3. 데이터 로드 실행

In [4]:
data = load_modeling_dataset()

print("최종 data shape:", data.shape)
display(data.head())

임차택시 원본 shape: (328875, 28)
특장차 원본 shape: (1393534, 51)
임차택시 바로콜 승차완료: (305729, 29)
특장차 바로콜 승차완료: (1099084, 52)


/var/folders/br/x7f7fw1907z6wpb38fpn1kpm0000gn/T/ipykernel_34292/3787358709.py:189: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = data.select_dtypes(include="object").columns


최종 data shape: (1404813, 67)


,_원자료_index,model_group,대기시간분석_제외사유,대기시간분석_포함여부,동일패턴건수_보완,목적구,목적동,배차_승차_분,배차_취소_분,배차일시,세부이동유형,승차거리,승차거리_km,승차거리구간,승차일시,심야시간사전예약_후보여부,예약목적여부,예정_배차_분,예정_승차_분,예정시,예정시간,예정일시,예정일자,요금,이용목적,임차택시_바로콜여부,임차택시_예약성예외여부,임차택시_장시간예외여부,임차택시_취소분석유형,장애유형,전일접수_후보여부,접수_배차_분,접수_승차_분,접수_취소_분,접수승차_날짜차이,접수시,접수시간대,접수시간대_HH,접수요일,접수일시,접수일자,정기접수_가능패턴여부,정기접수_목적후보여부,차량구분,출발구,출발동,취소_접수유형_후보,취소일시,취소일자,특장차_바로콜_후보여부,특장차_접수유형,특장차_접수유형_메모,특장차_접수유형_분류상태,특장차_접수유형_후보_보완,특장차_접수유형_후보_최종,특장차_탑승완료_시간논리정상여부,특장차_탑승완료_필수일시존재여부,평일주말,하차일시,target_min,hour,dayofweek,month,is_weekend,is_night,is_dawn_02_06,is_commute
0,0.0,특장차_바로콜,미상,미상,0.0,강북구,수유제2동,22.361000,NaN,2025-01-01 00:27:31.307,구 간 이동,17667,17.667,15~25km,2025-01-01 00:49:52.967,False,미상,26.086333,48.447333,0.0,00:01:26,2025-01-01 00:01:26.127,2025-01-01 00:00:00,3400.0,기타,미상,미상,미상,미상,지체,False,26.086333,48.447333,NaN,0.0,0.0,00:00:00,0.0,수,2025-01-01 00:01:26.127,2025-01-01 00:00:00,False,False,특장차,용산구,남영동,취소 아님,NaT,미상,True,NaN,NaN,미분류,바로콜 후보,바로콜 후보,True,True,평일,2025-01-01 01:35:06.747,48.447333,0,2,1,0,1,0,0
1,2.0,특장차_바로콜,미상,미상,0.0,노원구,하계1동,16.608883,NaN,2025-01-01 00:19:41.900,구 내 이동,2910,2.910,0.5~3km,2025-01-01 00:36:18.433,False,미상,15.698333,32.307217,0.0,00:04:00,2025-01-01 00:04:00.000,2025-01-01 00:00:00,1500.0,기타,미상,미상,미상,미상,뇌병,False,16.177833,32.786717,NaN,0.0,0.0,00:00:00,0.0,수,2025-01-01 00:03:31.230,2025-01-01 00:00:00,False,False,특장차,노원구,상계5동,취소 아님,NaT,미상,True,NaN,NaN,미분류,바로콜 후보,바로콜 후보,True,True,평일,2025-01-01 00:49:45.857,32.786717,0,2,1,0,1,0,0
2,3.0,특장차_바로콜,미상,미상,0.0,중구,명동,20.113167,NaN,2025-01-01 00:33:41.650,구 간 이동,10421,10.421,10~15km,2025-01-01 00:53:48.440,False,미상,29.694167,49.807333,0.0,00:04:00,2025-01-01 00:04:00.000,2025-01-01 00:00:00,2900.0,기타,미상,미상,미상,미상,지체,False,30.017950,50.131117,NaN,0.0,0.0,00:00:00,0.0,수,2025-01-01 00:03:40.573,2025-01-01 00:00:00,False,False,특장차,영등포구,당산제1동,취소 아님,NaT,미상,True,NaN,NaN,미분류,바로콜 후보,바로콜 후보,True,True,평일,2025-01-01 01:33:38.120,50.131117,0,2,1,0,1,0,0
3,4.0,특장차_바로콜,미상,미상,0.0,서초구,내곡동,33.567500,NaN,2025-01-01 00:06:33.790,구 간 이동,11943,11.943,10~15km,2025-01-01 00:40:07.840,False,미상,2.311883,35.879383,0.0,00:04:15,2025-01-01 00:04:15.077,2025-01-01 00:00:00,3000.0,기타,미상,미상,미상,미상,지체,False,2.311883,35.879383,NaN,0.0,0.0,00:00:00,0.0,수,2025-01-01 00:04:15.077,2025-01-01 00:00:00,False,False,특장차,송파구,오금동,취소 아님,NaT,미상,True,NaN,NaN,미분류,바로콜 후보,바로콜 후보,True,True,평일,2025-01-01 01:10:15.110,35.879383,0,2,1,0,1,0,0
4,5.0,특장차_바로콜,미상,미상,1.0,서초구,반포2동,38.008050,NaN,2025-01-01 00:16:19.457,구 내 이동,5205,5.205,5~10km,2025-01-01 00:54:19.940,False,미상,10.324283,48.332333,0.0,00:06:00,2025-01-01 00:06:00.000,2025-01-01 00:00:00,1700.0,귀가,미상,미상,미상,미상,뇌병,False,10.424283,48.432333,NaN,0.0,0.0,00:00:00,0.0,수,2025-01-01 00:05:54.000,2025-01-01 00:00:00,False,True,특장차,서초구,서초3동,취소 아님,NaT,미상,True,NaN,NaN,미분류,바로콜 후보,바로콜 후보,True,True,평일,2025-01-01 01:11:02.643,48.432333,0,2,1,0,1,0,0


In [5]:
display(
    data.groupby("model_group")["target_min"]
    .agg(
        건수="count",
        평균="mean",
        중앙값="median",
        p75=lambda x: x.quantile(0.75),
        p90=lambda x: x.quantile(0.90),
        p95=lambda x: x.quantile(0.95),
        p99=lambda x: x.quantile(0.99),
        최댓값="max",
    )
    .round(2)
)

,건수,평균,중앙값,p75,p90,p95,p99,최댓값
model_group,,,,,,,,
임차택시_바로콜,305729,42.45,28.51,50.28,87.34,145.05,174.64,249.90
특장차_바로콜,1099084,46.27,33.42,57.33,95.13,133.94,166.64,199.99


## 4. 일반구간 모델링 데이터 생성

전체 데이터는 삭제하지 않고 보존한다.

다만 일반적인 대기시간 예측 모델은 아래 구간을 학습 대상에서 제외한 데이터로 만든다.

```text
제외 조건
1. 접수→승차 대기시간이 130분 초과
2. 접수시간대가 02~06시

In [6]:
# =========================
# 4. 일반구간 / 별도관리 데이터 분리
# =========================

long_wait_cutoff = 130

is_long_tail = data["target_min"].gt(long_wait_cutoff)
is_dawn_02_06 = data["hour"].between(2, 6, inclusive="both")

general_mask = ~is_long_tail & ~is_dawn_02_06

general_model_data = data[general_mask].copy()
excluded_model_data = data[~general_mask].copy()

print("전체 데이터:", data.shape)
print("일반구간 모델링 데이터:", general_model_data.shape)
print("별도관리 데이터:", excluded_model_data.shape)
print("별도관리 비율(%):", round(len(excluded_model_data) / len(data) * 100, 2))

전체 데이터: (1404813, 67)
일반구간 모델링 데이터: (1293478, 67)
별도관리 데이터: (111335, 67)
별도관리 비율(%): 7.93


In [7]:
# =========================
# 4-1. 제외 조건별 건수 확인
# =========================

exclude_summary = pd.Series({
    "전체 건수": len(data),
    "130분 초과 건수": is_long_tail.sum(),
    "02~06시 건수": is_dawn_02_06.sum(),
    "130분 초과 & 02~06시 건수": (is_long_tail & is_dawn_02_06).sum(),
    "130분 초과 비율(%)": is_long_tail.mean() * 100,
    "02~06시 비율(%)": is_dawn_02_06.mean() * 100,
    "겹침 비율 - 전체 대비(%)": (is_long_tail & is_dawn_02_06).mean() * 100,
    "별도관리 전체 비율(%)": (~general_mask).mean() * 100,
})

exclude_summary.round(2)

전체 건수                  1404813.00
130분 초과 건수               78429.00
02~06시 건수                62985.00
130분 초과 & 02~06시 건수      30079.00
130분 초과 비율(%)                5.58
02~06시 비율(%)                 4.48
겹침 비율 - 전체 대비(%)             2.14
별도관리 전체 비율(%)                7.93
dtype: float64

In [8]:
# =========================
# 4-2. 차량유형별 일반구간 / 별도관리 구분 확인
# =========================

data_exclude_check = data.copy()

data_exclude_check["모델링구분"] = np.where(
    general_mask,
    "일반구간_모델링",
    "별도관리_제외",
)

data_exclude_check["제외사유"] = np.select(
    [
        is_long_tail & is_dawn_02_06,
        is_long_tail,
        is_dawn_02_06,
    ],
    [
        "130분초과_AND_02~06시",
        "130분초과",
        "02~06시",
    ],
    default="일반구간",
)

exclude_by_group = (
    data_exclude_check
    .groupby(["model_group", "제외사유"])
    .agg(
        건수=("target_min", "size"),
        평균=("target_min", "mean"),
        중앙값=("target_min", "median"),
        p90=("target_min", lambda x: x.quantile(0.90)),
    )
    .reset_index()
)

exclude_by_group["전체대비비율(%)"] = exclude_by_group["건수"] / len(data) * 100

display(exclude_by_group.round(2))

,model_group,제외사유,건수,평균,중앙값,p90,전체대비비율(%)
0,임차택시_바로콜,02~06시,6667,58.32,55.86,96.93,0.47
1,임차택시_바로콜,130분초과,10767,151.95,148.25,171.70,0.77
2,임차택시_바로콜,130분초과_AND_02~06시,10047,162.81,158.17,192.16,0.72
3,임차택시_바로콜,일반구간,278248,33.49,26.61,65.97,19.81
4,특장차_바로콜,02~06시,26239,59.67,53.13,110.85,1.87
5,특장차_바로콜,130분초과,37583,151.28,149.14,169.18,2.68
6,특장차_바로콜,130분초과_AND_02~06시,20032,160.92,158.55,184.48,1.43
7,특장차_바로콜,일반구간,1015230,39.77,31.72,78.40,72.27


In [9]:
# =========================
# 4-3. 일반구간 모델링 데이터 분포 확인
# =========================

display(
    general_model_data
    .groupby("model_group")["target_min"]
    .agg(
        건수="count",
        평균="mean",
        중앙값="median",
        p75=lambda x: x.quantile(0.75),
        p90=lambda x: x.quantile(0.90),
        p95=lambda x: x.quantile(0.95),
        최댓값="max",
    )
    .round(2)
)

,건수,평균,중앙값,p75,p90,p95,최댓값
model_group,,,,,,,
임차택시_바로콜,278248,33.49,26.61,42.41,65.97,79.68,129.98
특장차_바로콜,1015230,39.77,31.72,50.62,78.40,94.12,130.00


## 5. model_group별 출발구 공급 흐름 proxy 생성

실시간 차량 위치 데이터가 없기 때문에, 승차/하차 이력을 이용해 지역별 공급 흐름을 근사한다.

현재 호출의 기준은 다음과 같다.

```text
접수시각 = t
출발구 = g
차량유형 = m
```

생성할 피처는 다음 6개다.

| 피처 | 의미 |
|---|---|
| `model_group_origin_gu_dropoff_count_prev_30m` | 같은 차량유형이 최근 30분 동안 현재 출발구에서 하차한 건수 |
| `model_group_origin_gu_dropoff_count_prev_60m` | 같은 차량유형이 최근 60분 동안 현재 출발구에서 하차한 건수 |
| `model_group_origin_gu_pickup_count_prev_30m` | 같은 차량유형이 최근 30분 동안 현재 출발구에서 승차한 건수 |
| `model_group_origin_gu_pickup_count_prev_60m` | 같은 차량유형이 최근 60분 동안 현재 출발구에서 승차한 건수 |
| `model_group_origin_gu_net_supply_prev_30m` | 최근 30분 하차 건수 - 승차 건수 |
| `model_group_origin_gu_net_supply_prev_60m` | 최근 60분 하차 건수 - 승차 건수 |

이 피처는 반드시 현재 `접수일시` 이전에 발생한 `승차일시`, `하차일시`만 사용한다.

In [10]:
# =========================
# 5-1. 시간창 기반 event count 함수
# =========================


def add_prev_event_count_by_model_group_region(
    target_df,
    event_df,
    event_time_col,
    event_region_col,
    output_col,
    window_minutes,
):
    """현재 접수시각 이전 window 안의 같은 model_group×지역 event 건수를 계산한다."""
    result = target_df.copy()
    result["_row_id"] = np.arange(len(result))
    result[output_col] = 0

    query = (
        result[["_row_id", "접수일시", "model_group", "출발구"]]
        .rename(columns={"출발구": "region"})
        .dropna(subset=["접수일시", "model_group", "region"])
    )

    events = (
        event_df[["model_group", event_region_col, event_time_col]]
        .rename(columns={event_region_col: "region", event_time_col: "event_time"})
        .dropna(subset=["model_group", "region", "event_time"])
    )

    events["event_time"] = pd.to_datetime(events["event_time"], errors="coerce")
    events = events.dropna(subset=["event_time"])

    event_time_dict = {}
    for key, group in events.groupby(["model_group", "region"], observed=True):
        event_time_dict[key] = np.sort(
            pd.to_datetime(group["event_time"]).to_numpy(dtype="datetime64[ns]")
        )

    window_delta = np.timedelta64(window_minutes, "m")

    for key, group in query.groupby(["model_group", "region"], observed=True):
        event_times = event_time_dict.get(key)
        if event_times is None or len(event_times) == 0:
            continue

        query_times = pd.to_datetime(group["접수일시"]).to_numpy(dtype="datetime64[ns]")
        start_times = query_times - window_delta

        right_idx = np.searchsorted(event_times, query_times, side="left")
        left_idx = np.searchsorted(event_times, start_times, side="left")
        counts = right_idx - left_idx

        result.loc[group["_row_id"].to_numpy(), output_col] = counts

    result = result.drop(columns=["_row_id"])
    return result


In [11]:
# =========================
# 5-2. model_group별 공급 흐름 proxy 생성
# =========================

supply_proxy_data = general_model_data.sort_values("접수일시").reset_index(drop=True).copy()

# event 후보는 전체 승차완료 데이터(data)를 사용한다.
# 단, 각 행의 proxy 계산에서는 접수일시 이전 event만 count하므로 미래 event는 포함되지 않는다.
event_source = data.sort_values("접수일시").reset_index(drop=True).copy()

for minutes in [30, 60]:
    dropoff_col = f"model_group_origin_gu_dropoff_count_prev_{minutes}m"
    pickup_col = f"model_group_origin_gu_pickup_count_prev_{minutes}m"
    net_col = f"model_group_origin_gu_net_supply_prev_{minutes}m"

    supply_proxy_data = add_prev_event_count_by_model_group_region(
        target_df=supply_proxy_data,
        event_df=event_source,
        event_time_col="하차일시",
        event_region_col="목적구",
        output_col=dropoff_col,
        window_minutes=minutes,
    )

    supply_proxy_data = add_prev_event_count_by_model_group_region(
        target_df=supply_proxy_data,
        event_df=event_source,
        event_time_col="승차일시",
        event_region_col="출발구",
        output_col=pickup_col,
        window_minutes=minutes,
    )

    supply_proxy_data[net_col] = supply_proxy_data[dropoff_col] - supply_proxy_data[pickup_col]

SUPPLY_FLOW_FEATURES = [
    "model_group_origin_gu_dropoff_count_prev_30m",
    "model_group_origin_gu_dropoff_count_prev_60m",
    "model_group_origin_gu_pickup_count_prev_30m",
    "model_group_origin_gu_pickup_count_prev_60m",
    "model_group_origin_gu_net_supply_prev_30m",
    "model_group_origin_gu_net_supply_prev_60m",
]

print("supply_proxy_data shape:", supply_proxy_data.shape)
print("생성된 공급 흐름 피처:", SUPPLY_FLOW_FEATURES)

display(supply_proxy_data[["접수일시", "model_group", "출발구"] + SUPPLY_FLOW_FEATURES + ["target_min"]].head())
display(supply_proxy_data[SUPPLY_FLOW_FEATURES].describe().round(2))


supply_proxy_data shape: (1293478, 73)
생성된 공급 흐름 피처: ['model_group_origin_gu_dropoff_count_prev_30m', 'model_group_origin_gu_dropoff_count_prev_60m', 'model_group_origin_gu_pickup_count_prev_30m', 'model_group_origin_gu_pickup_count_prev_60m', 'model_group_origin_gu_net_supply_prev_30m', 'model_group_origin_gu_net_supply_prev_60m']


,접수일시,model_group,출발구,model_group_origin_gu_dropoff_count_prev_30m,model_group_origin_gu_dropoff_count_prev_60m,model_group_origin_gu_pickup_count_prev_30m,model_group_origin_gu_pickup_count_prev_60m,model_group_origin_gu_net_supply_prev_30m,model_group_origin_gu_net_supply_prev_60m,target_min
0,2025-01-01 00:01:26.127,특장차_바로콜,용산구,0,0,0,0,0,0,48.447333
1,2025-01-01 00:03:31.230,특장차_바로콜,노원구,0,0,0,0,0,0,32.786717
2,2025-01-01 00:03:40.573,특장차_바로콜,영등포구,0,0,0,0,0,0,50.131117
3,2025-01-01 00:04:15.077,특장차_바로콜,송파구,0,0,0,0,0,0,35.879383
4,2025-01-01 00:05:54.000,특장차_바로콜,서초구,0,0,0,0,0,0,48.432333


,model_group_origin_gu_dropoff_count_prev_30m,model_group_origin_gu_dropoff_count_prev_60m,model_group_origin_gu_pickup_count_prev_30m,model_group_origin_gu_pickup_count_prev_60m,model_group_origin_gu_net_supply_prev_30m,model_group_origin_gu_net_supply_prev_60m
count,1293478.00,1293478.00,1293478.00,1293478.00,1293478.00,1293478.00
mean,5.43,10.64,6.04,11.80,-0.60,-1.16
std,5.08,9.56,5.49,10.29,3.37,4.94
min,0.00,0.00,0.00,0.00,-32.00,-43.00
25%,2.00,4.00,2.00,4.00,-2.00,-4.00
50%,4.00,8.00,5.00,9.00,0.00,-1.00
75%,8.00,15.00,8.00,16.00,1.00,1.00
max,43.00,76.00,48.00,74.00,24.00,42.00


## 6. Time-based train / validation / test 분리

기존 모델링은 `month × model_group` 기준 random split이었다.

이번 실험은 공급 흐름 proxy가 시간 순서에 의존하므로, 실제 운영 환경에 가깝게 `접수일시` 기준 time-based split을 사용한다.

기본 분리 기준은 다음과 같다.

```text
train: 2025-01-01 ~ 2025-09-30
valid: 2025-10-01 ~ 2025-10-31
test : 2025-11-01 ~ 2025-12-31
```

데이터 기간이 다르면 아래 날짜를 조정하면 된다.

In [12]:
# =========================
# 6-1. time-based split
# =========================

TIME_TRAIN_END = pd.Timestamp("2025-10-01")
TIME_VALID_END = pd.Timestamp("2025-11-01")

supply_proxy_data = supply_proxy_data.sort_values("접수일시").reset_index(drop=True)

train = supply_proxy_data[supply_proxy_data["접수일시"] < TIME_TRAIN_END].copy()
valid = supply_proxy_data[
    (supply_proxy_data["접수일시"] >= TIME_TRAIN_END)
    & (supply_proxy_data["접수일시"] < TIME_VALID_END)
].copy()
test = supply_proxy_data[supply_proxy_data["접수일시"] >= TIME_VALID_END].copy()

print("train:", train.shape, train["접수일시"].min(), "~", train["접수일시"].max())
print("valid:", valid.shape, valid["접수일시"].min(), "~", valid["접수일시"].max())
print("test:", test.shape, test["접수일시"].min(), "~", test["접수일시"].max())

if min(len(train), len(valid), len(test)) == 0:
    raise ValueError("time-based split 결과 비어 있는 split이 있습니다. TIME_TRAIN_END / TIME_VALID_END를 조정하세요.")


train: (972543, 73) 2025-01-01 00:01:26.127000 ~ 2025-09-30 23:50:09.617000
valid: (103832, 73) 2025-10-01 00:00:32.890000 ~ 2025-10-31 23:59:09
test: (217103, 73) 2025-11-01 00:00:14 ~ 2025-12-31 23:52:59


In [13]:
# =========================
# 6-2. split 결과 요약
# =========================

def split_summary(frame, name):
    return {
        "split": name,
        "rows": len(frame),
        "target_mean": frame["target_min"].mean(),
        "target_median": frame["target_min"].median(),
        "target_p90": frame["target_min"].quantile(0.90),
        "target_max": frame["target_min"].max(),
        "min_date": frame["접수일시"].min(),
        "max_date": frame["접수일시"].max(),
    }

split_summary_df = pd.DataFrame([
    split_summary(train, "train"),
    split_summary(valid, "valid"),
    split_summary(test, "test"),
])

display(split_summary_df.round(4))

model_group_ratio = pd.concat(
    [
        train["model_group"].value_counts(normalize=True).rename("train"),
        valid["model_group"].value_counts(normalize=True).rename("valid"),
        test["model_group"].value_counts(normalize=True).rename("test"),
    ],
    axis=1,
)

display(model_group_ratio.round(4))


/var/folders/br/x7f7fw1907z6wpb38fpn1kpm0000gn/T/ipykernel_34292/4104820584.py:23: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(split_summary_df.round(4))


,split,rows,target_mean,target_median,target_p90,target_max,min_date,max_date
0,train,972543,37.6101,30.0303,73.8814,129.9986,2025-01-01 00:01:26.127,2025-09-30 23:50:09.617
1,valid,103832,39.9560,31.8461,78.3476,129.9758,2025-10-01 00:00:32.890,2025-10-31 23:59:09.000
2,test,217103,41.3230,32.8327,81.8563,129.9902,2025-11-01 00:00:14.000,2025-12-31 23:52:59.000


,train,valid,test
model_group,,,
특장차_바로콜,0.7912,0.766,0.7655
임차택시_바로콜,0.2088,0.234,0.2345


## 7. 전일 차량운행 공급 proxy 추가

기존 실험에서 전일 차량운행 대수는 성능 개선 효과가 있었고, 예측 시점 이전에 알 수 있는 값이라 leakage 위험이 낮다.

이번 time-based 실험에서도 `vehicle_operation_count_prev_day`를 공급 proxy로 추가한다.

In [14]:
# =========================
# 7-1. 전일 차량운행 데이터 merge
# =========================

DAILY_USAGE_FILENAME = "서울시설공단_장애인콜택시 일별이용현황_20251231.csv"
DAILY_USAGE_PATH = find_csv_in_processed(DATA_DIR, DAILY_USAGE_FILENAME)

print("DAILY_USAGE_PATH:", DAILY_USAGE_PATH)

daily_usage = pd.read_csv(DAILY_USAGE_PATH, low_memory=False)
daily_usage["기준일"] = pd.to_datetime(daily_usage["기준일"], errors="coerce").dt.normalize()
daily_usage["차량운행"] = pd.to_numeric(daily_usage["차량운행"], errors="coerce")

vehicle_operation_today_table = (
    daily_usage
    .dropna(subset=["기준일"])
    .groupby("기준일", as_index=False)
    .agg(vehicle_operation_count_today=("차량운행", "first"))
)

PREV_DAY_VEHICLE_OPERATION_COL = "vehicle_operation_count_prev_day"

vehicle_operation_prev_day_table = vehicle_operation_today_table.copy()
vehicle_operation_prev_day_table["기준일"] = vehicle_operation_prev_day_table["기준일"] + pd.Timedelta(days=1)
vehicle_operation_prev_day_table = vehicle_operation_prev_day_table.rename(
    columns={"vehicle_operation_count_today": PREV_DAY_VEHICLE_OPERATION_COL}
)


def add_prev_day_vehicle_operation_count(frame, prev_day_table, fallback_value=None):
    tmp = frame.copy()
    tmp["_merge_기준일"] = tmp["접수일시"].dt.normalize()

    tmp = tmp.merge(
        prev_day_table,
        left_on="_merge_기준일",
        right_on="기준일",
        how="left",
    )

    if fallback_value is None:
        fallback_value = tmp[PREV_DAY_VEHICLE_OPERATION_COL].median()

    missing_count = tmp[PREV_DAY_VEHICLE_OPERATION_COL].isna().sum()
    tmp[PREV_DAY_VEHICLE_OPERATION_COL] = tmp[PREV_DAY_VEHICLE_OPERATION_COL].fillna(fallback_value)
    tmp = tmp.drop(columns=["_merge_기준일", "기준일"], errors="ignore")

    return tmp, missing_count, fallback_value

train, train_missing, train_vehicle_fallback = add_prev_day_vehicle_operation_count(
    train,
    vehicle_operation_prev_day_table,
    fallback_value=None,
)
valid, valid_missing, _ = add_prev_day_vehicle_operation_count(
    valid,
    vehicle_operation_prev_day_table,
    fallback_value=train_vehicle_fallback,
)
test, test_missing, _ = add_prev_day_vehicle_operation_count(
    test,
    vehicle_operation_prev_day_table,
    fallback_value=train_vehicle_fallback,
)

print("전일 차량운행 결측 보완 전:")
print("train:", train_missing)
print("valid:", valid_missing)
print("test:", test_missing)
print("fallback value:", train_vehicle_fallback)

display(train[["접수일시", "model_group", PREV_DAY_VEHICLE_OPERATION_COL, "target_min"]].head())


DAILY_USAGE_PATH: /Users/blaumonde/calltaxi-DA/data/processed/서울시설공단_장애인콜택시 일별이용현황_20251231.csv
전일 차량운행 결측 보완 전:
train: 1060
valid: 0
test: 0
fallback value: 681.0


,접수일시,model_group,vehicle_operation_count_prev_day,target_min
0,2025-01-01 00:01:26.127,특장차_바로콜,681.0,48.447333
1,2025-01-01 00:03:31.230,특장차_바로콜,681.0,32.786717
2,2025-01-01 00:03:40.573,특장차_바로콜,681.0,50.131117
3,2025-01-01 00:04:15.077,특장차_바로콜,681.0,35.879383
4,2025-01-01 00:05:54.000,특장차_바로콜,681.0,48.432333


## 8. Feature Set 정의

이번 실험에서는 같은 time-based split 안에서 3개 피처셋을 비교한다.

1. `FEATURES_TS_BASE`: 기존 Feature Set v2
2. `FEATURES_TS_PREV_DAY`: 기존 피처 + 전일 차량운행
3. `FEATURES_TS_SUPPLY_FLOW`: 기존 피처 + 전일 차량운행 + 공급 흐름 proxy 6개

In [15]:
# =========================
# 8-1. Feature Set 정의
# =========================

BASE_FEATURES = [
    "hour",
    "이용목적",
    "승차거리",
    "출발동",
    "목적동",
    "출발구",
    "목적구",
    "month",
    "세부이동유형",
    "dayofweek",
    "model_group",
]

DAILY_SUPPLY_FEATURES = [
    PREV_DAY_VEHICLE_OPERATION_COL,
]

FEATURES_TS_BASE = BASE_FEATURES
FEATURES_TS_PREV_DAY = BASE_FEATURES + DAILY_SUPPLY_FEATURES
FEATURES_TS_SUPPLY_FLOW = BASE_FEATURES + DAILY_SUPPLY_FEATURES + SUPPLY_FLOW_FEATURES

TARGET_COL = "target_min"

for frame in [train, valid, test]:
    frame["승차거리"] = pd.to_numeric(frame["승차거리"], errors="coerce")

for feature_set_name, features in [
    ("FEATURES_TS_BASE", FEATURES_TS_BASE),
    ("FEATURES_TS_PREV_DAY", FEATURES_TS_PREV_DAY),
    ("FEATURES_TS_SUPPLY_FLOW", FEATURES_TS_SUPPLY_FLOW),
]:
    missing = [col for col in features if col not in train.columns]
    if missing:
        raise ValueError(f"{feature_set_name}에 train에 없는 피처가 있습니다: {missing}")
    print(feature_set_name, len(features), "개")

print("최종 공급 흐름 포함 피처:")
print(FEATURES_TS_SUPPLY_FLOW)


FEATURES_TS_BASE 11 개
FEATURES_TS_PREV_DAY 12 개
FEATURES_TS_SUPPLY_FLOW 18 개
최종 공급 흐름 포함 피처:
['hour', '이용목적', '승차거리', '출발동', '목적동', '출발구', '목적구', 'month', '세부이동유형', 'dayofweek', 'model_group', 'vehicle_operation_count_prev_day', 'model_group_origin_gu_dropoff_count_prev_30m', 'model_group_origin_gu_dropoff_count_prev_60m', 'model_group_origin_gu_pickup_count_prev_30m', 'model_group_origin_gu_pickup_count_prev_60m', 'model_group_origin_gu_net_supply_prev_30m', 'model_group_origin_gu_net_supply_prev_60m']


## 9. 평가 함수 정의

In [16]:
# =========================
# 9-1. 평가 함수
# =========================

from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "Median_AE": median_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }


def risk_metrics(y_true, y_pred, threshold):
    actual = y_true >= threshold
    pred = y_pred >= threshold
    return {
        "threshold": threshold,
        "Accuracy": accuracy_score(actual, pred),
        "Precision": precision_score(actual, pred, zero_division=0),
        "Recall": recall_score(actual, pred, zero_division=0),
        "F1": f1_score(actual, pred, zero_division=0),
        "Confusion_Matrix": confusion_matrix(actual, pred, labels=[False, True]).tolist(),
    }


def evaluate_predictions(frame, pred, threshold):
    y = frame[TARGET_COL].to_numpy()
    return {
        **regression_metrics(y, pred),
        **{f"risk_{k}": v for k, v in risk_metrics(y, pred, threshold).items()},
    }


def evaluate_by_group(frame, pred, threshold_by_group):
    tmp = frame[["model_group", TARGET_COL]].copy()
    tmp["prediction"] = pred

    rows = []
    for group, group_df in tmp.groupby("model_group"):
        y = group_df[TARGET_COL].to_numpy()
        p = group_df["prediction"].to_numpy()
        threshold = threshold_by_group[group]

        reg = regression_metrics(y, p)
        risk = risk_metrics(y, p, threshold)

        rows.append({
            "model_group": group,
            "rows": len(group_df),
            **reg,
            "risk_threshold": threshold,
            "risk_Accuracy": risk["Accuracy"],
            "risk_Precision": risk["Precision"],
            "risk_Recall": risk["Recall"],
            "risk_F1": risk["F1"],
        })

    return pd.DataFrame(rows)


## 10. RandomForest 모델 학습 및 비교

RandomForest는 여러 개의 결정트리를 학습한 뒤 평균 예측값을 사용하는 앙상블 모델이다.

이번 실험에서는 기존 time-based split과 공급 흐름 proxy 구조를 그대로 유지하고, 모델만 RandomForest로 바꿔 성능을 비교한다.

범주형 변수는 `OrdinalEncoder`로 정수 인코딩하고, 수치형 변수는 그대로 사용한다.


In [17]:
# =========================
# 10-1. RandomForestRegressor Pipeline 구성
# =========================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

CATEGORICAL_COLS_TS = [
    "이용목적",
    "출발동",
    "목적동",
    "출발구",
    "목적구",
    "세부이동유형",
    "model_group",
]

BASE_NUMERIC_COLS_TS = [
    "hour",
    "승차거리",
    "month",
    "dayofweek",
]


def build_rf_ts_pipeline(numeric_cols, params=None):
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "cat",
                OrdinalEncoder(
                    handle_unknown="use_encoded_value",
                    unknown_value=-1,
                    encoded_missing_value=-1,
                ),
                CATEGORICAL_COLS_TS,
            ),
            (
                "num",
                "passthrough",
                numeric_cols,
            ),
        ],
        verbose_feature_names_out=False,
        remainder="drop",
    )

    default_params = {
        "n_estimators": 100,
        "max_depth": 26,
        "min_samples_leaf": 5,
        "max_features": 0.8,
        "n_jobs": -1,
        "random_state": RANDOM_STATE,
        "verbose": 1,
    }

    if params:
        default_params.update(params)

    return Pipeline([
        ("preprocess", preprocessor),
        ("model", RandomForestRegressor(**default_params)),
    ])


In [18]:
# =========================
# 10-2. 피처셋별 RandomForest 학습 및 평가
# =========================

long_wait_threshold_ts = train[TARGET_COL].quantile(0.90)
threshold_by_group_ts = train.groupby("model_group")[TARGET_COL].quantile(0.90).to_dict()

print("장시간 대기 기준 train p90:", round(long_wait_threshold_ts, 4))
print("차량유형별 p90:", {k: round(v, 4) for k, v in threshold_by_group_ts.items()})

experiment_configs = [
    {
        "feature_set": "TS_BASE",
        "features": FEATURES_TS_BASE,
        "numeric_cols": BASE_NUMERIC_COLS_TS,
    },
    {
        "feature_set": "TS_PREV_DAY_VEHICLE",
        "features": FEATURES_TS_PREV_DAY,
        "numeric_cols": BASE_NUMERIC_COLS_TS + DAILY_SUPPLY_FEATURES,
    },
    {
        "feature_set": "TS_PREV_DAY_VEHICLE_SUPPLY_FLOW",
        "features": FEATURES_TS_SUPPLY_FLOW,
        "numeric_cols": BASE_NUMERIC_COLS_TS + DAILY_SUPPLY_FEATURES + SUPPLY_FLOW_FEATURES,
    },
]

rf_ts_models = {}
rf_ts_predictions = {}
rf_ts_result_rows = []

for config in experiment_configs:
    feature_set = config["feature_set"]
    features = config["features"]
    numeric_cols = config["numeric_cols"]

    print("\n===", feature_set, "===")
    print("피처 수:", len(features))

    model = build_rf_ts_pipeline(numeric_cols=numeric_cols)
    model.fit(train[features], train[TARGET_COL])

    valid_pred = model.predict(valid[features])
    test_pred = model.predict(test[features])

    valid_metrics = evaluate_predictions(valid, valid_pred, threshold=long_wait_threshold_ts)
    test_metrics = evaluate_predictions(test, test_pred, threshold=long_wait_threshold_ts)

    rf_ts_models[feature_set] = model
    rf_ts_predictions[(feature_set, "valid")] = valid_pred
    rf_ts_predictions[(feature_set, "test")] = test_pred

    rf_ts_result_rows.append({
        "feature_set": feature_set,
        "model": "RandomForestRegressor",
        "split": "valid",
        **valid_metrics,
    })
    rf_ts_result_rows.append({
        "feature_set": feature_set,
        "model": "RandomForestRegressor",
        "split": "test",
        **test_metrics,
    })

rf_ts_results = pd.DataFrame(rf_ts_result_rows)

display(
    rf_ts_results
    .drop(columns=["risk_Confusion_Matrix"], errors="ignore")
    .sort_values(["split", "MAE"])
    .round(4)
)


장시간 대기 기준 train p90: 73.8814
차량유형별 p90: {'임차택시_바로콜': 64.4348, '특장차_바로콜': 76.5326}

=== TS_BASE ===
피처 수: 11


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   17.7s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   45.0s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.1s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.3s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.2s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.5s finished



=== TS_PREV_DAY_VEHICLE ===
피처 수: 12


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   19.2s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   50.4s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.1s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.3s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.2s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.5s finished



=== TS_PREV_DAY_VEHICLE_SUPPLY_FLOW ===
피처 수: 18


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   31.6s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:  1.3min finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.1s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.3s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.2s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.5s finished


,feature_set,model,split,MAE,Median_AE,RMSE,R2,risk_threshold,risk_Accuracy,risk_Precision,risk_Recall,risk_F1
1,TS_BASE,RandomForestRegressor,test,13.7477,9.7487,18.9436,0.4653,73.8814,0.8839,0.6234,0.3039,0.4087
5,TS_PREV_DAY_VEHICLE_SUPPLY_FLOW,RandomForestRegressor,test,14.1047,10.1668,19.2746,0.4464,73.8814,0.8821,0.6221,0.2719,0.3785
3,TS_PREV_DAY_VEHICLE,RandomForestRegressor,test,14.3744,10.4182,19.5848,0.4285,73.8814,0.8802,0.5957,0.2869,0.3873
4,TS_PREV_DAY_VEHICLE_SUPPLY_FLOW,RandomForestRegressor,valid,14.1081,10.2550,19.1108,0.4231,73.8814,0.8955,0.6028,0.3068,0.4066
0,TS_BASE,RandomForestRegressor,valid,14.1736,10.1781,19.2584,0.4141,73.8814,0.8935,0.5722,0.3466,0.4317
2,TS_PREV_DAY_VEHICLE,RandomForestRegressor,valid,14.2135,10.2576,19.2908,0.4122,73.8814,0.8930,0.5735,0.3249,0.4148


## 11. 성능 개선폭 확인

같은 time-based split 안에서 기존 피처셋 대비 전일 차량운행과 공급 흐름 proxy가 성능을 개선했는지 확인한다.

In [19]:
# =========================
# 11-1. test 기준 개선폭 확인
# =========================

baseline_test = rf_ts_results.query("split == 'test' and feature_set == 'TS_BASE'").iloc[0]

summary_rows = []
for feature_set in ["TS_PREV_DAY_VEHICLE", "TS_PREV_DAY_VEHICLE_SUPPLY_FLOW"]:
    row = rf_ts_results.query("split == 'test' and feature_set == @feature_set").iloc[0]
    for metric in ["MAE", "Median_AE", "RMSE", "R2", "risk_F1"]:
        summary_rows.append({
            "feature_set": feature_set,
            "metric": metric,
            "baseline": baseline_test[metric],
            "proxy_model": row[metric],
            "diff_proxy_minus_baseline": row[metric] - baseline_test[metric],
        })

rf_ts_improvement_summary = pd.DataFrame(summary_rows)

display(rf_ts_improvement_summary.round(4))


,feature_set,metric,baseline,proxy_model,diff_proxy_minus_baseline
0,TS_PREV_DAY_VEHICLE,MAE,13.7477,14.3744,0.6267
1,TS_PREV_DAY_VEHICLE,Median_AE,9.7487,10.4182,0.6695
2,TS_PREV_DAY_VEHICLE,RMSE,18.9436,19.5848,0.6413
3,TS_PREV_DAY_VEHICLE,R2,0.4653,0.4285,-0.0368
4,TS_PREV_DAY_VEHICLE,risk_F1,0.4087,0.3873,-0.0214
5,TS_PREV_DAY_VEHICLE_SUPPLY_FLOW,MAE,13.7477,14.1047,0.3570
6,TS_PREV_DAY_VEHICLE_SUPPLY_FLOW,Median_AE,9.7487,10.1668,0.4182
7,TS_PREV_DAY_VEHICLE_SUPPLY_FLOW,RMSE,18.9436,19.2746,0.3310
8,TS_PREV_DAY_VEHICLE_SUPPLY_FLOW,R2,0.4653,0.4464,-0.0188
9,TS_PREV_DAY_VEHICLE_SUPPLY_FLOW,risk_F1,0.4087,0.3785,-0.0302


## 12. 최종 후보 차량유형별 성능 확인

가장 실험 목적에 가까운 모델인 `TS_PREV_DAY_VEHICLE_SUPPLY_FLOW`의 차량유형별 성능을 확인한다.

In [20]:
# =========================
# 12-1. 공급 흐름 포함 모델 차량유형별 성능
# =========================

FINAL_FEATURE_SET = "TS_PREV_DAY_VEHICLE_SUPPLY_FLOW"
final_test_pred = rf_ts_predictions[(FINAL_FEATURE_SET, "test")]

group_metrics_rf_ts_supply_flow = evaluate_by_group(
    test,
    final_test_pred,
    threshold_by_group_ts,
)

display(group_metrics_rf_ts_supply_flow.round(4))

print("test confusion matrix:")
rf_ts_results.query("split == 'test' and feature_set == @FINAL_FEATURE_SET")["risk_Confusion_Matrix"].iloc[0]


,model_group,rows,MAE,Median_AE,RMSE,R2,risk_threshold,risk_Accuracy,risk_Precision,risk_Recall,risk_F1
0,임차택시_바로콜,50912,12.4035,8.9605,17.0022,0.4633,64.4348,0.8880,0.6073,0.3578,0.4503
1,특장차_바로콜,166191,14.6259,10.5834,19.9189,0.4307,76.5326,0.8794,0.6302,0.2449,0.3527


test confusion matrix:


[[183709, 4734], [20866, 7794]]

## 13. 해석 메모

이 실험은 기존 random split 모델과 직접 수치를 비교하기 위한 것이 아니라, 실제 운영 환경에 가까운 time-based split 안에서 공급 proxy 추가 효과를 확인하기 위한 실험이다.

해석 기준은 다음과 같다.

- `TS_BASE` 대비 `TS_PREV_DAY_VEHICLE`의 MAE가 감소하면 전일 차량운행 대수가 공급 proxy로 도움이 된 것이다.
- `TS_PREV_DAY_VEHICLE` 대비 `TS_PREV_DAY_VEHICLE_SUPPLY_FLOW`의 MAE가 추가로 감소하면, 최근 30분/60분의 지역별 차량 유입·유출 흐름이 대기시간 예측에 도움이 된 것이다.
- 공급 흐름 proxy는 임차택시와 특장차의 차량 자원이 다르다는 점을 고려해 `model_group`별로 분리해 계산하였다.